# 01 Wrangle

Loads the raw UniProt Excel file, adds derived variables, and saves a single clean dataset.

**Output:** `../output/full_dataset.csv` (also copied to `../dashboard/full_dataset.csv`)

| Column added | Source |
|---|---|
| `pLLPS_Class` | thresholds: High â‰¥ 0.7, Low < 0.4 |
| `GO_IDs` | UniProt REST (cached) |
| `Functional_Categories` | GO parent IDs expanded through GO DAG |
| `Functional Group` | first GO-based category, else 'Other' |
| `GO_Slim_Categories` | GO Slim mapping (generic) |
| `Functional Slim` | first GO Slim category, else 'Other' |
| `Location Categories` | UniProt subcell SL ontology terms |
| `Location_IDs` | UniProt SL accessions |
| `Location Primary` | first SL term, else 'Other' |
| `Is_Membrane` | SL hierarchy + TMD_count |
| `TMD_count` | transmembrane domain count (from cache) |

In [1]:
from pathlib import Path

import pandas as pd

from llps.data import (
    add_go_annotations,
    add_tmd_count,
    fetch_uniprot_go_annotations,
    fetch_uniprot_tm_annotations,
    load_llps_data,
)
from llps.functional import add_functional_categories, add_go_slim_categories

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]:
    if (candidate / "Human Phase separation data.xlsx").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root")

RAW = ROOT / "Human Phase separation data.xlsx"
CACHE = ROOT / "data/uniprot_tm_cache.csv"
GO_CACHE = ROOT / "data/uniprot_go_cache.csv"
OUT = ROOT / "output/full_dataset.csv"
APP_OUT = ROOT / "dashboard/full_dataset.csv"


In [2]:
df = load_llps_data(RAW)
print(f"{len(df):,} proteins  |  p(LLPS) {df['p(LLPS)'].min():.2f}â€“{df['p(LLPS)'].max():.2f}")
df.head(2)

✅ Loaded 20366 proteins from /workspaces/mem_prot_llps/Human Phase separation data.xlsx
   p(LLPS) range: 0.060 - 1.000
20,366 proteins  |  p(LLPS) 0.06â€“1.00


,Entry,Entry name,Protein names,p(LLPS),n(DPR=> 25),Organism,Length,Function [CC],Subcellular location [CC],Involvement in disease,Cross-reference (PDB)
0,Q9Y6V0,PCLO_HUMAN,Protein piccolo (Aczonin),1.0,21,Homo sapiens (Human),5142.0,Scaffold protein of the presynaptic cytomatri...,"Cell junction, synapse, presynaptic active zo...",Pontocerebellar hypoplasia 3 (PCH3) [MIM:6080...,1UJD;
1,Q9Y566,SHAN1_HUMAN,SH3 and multiple ankyrin repeat domains protei...,1.0,8,Homo sapiens (Human),2161.0,Seems to be an adapter protein in the postsyn...,"Cytoplasm {ECO:0000250}. Cell junction, synap...",NaN,6CPI;


In [3]:
# GO annotations (cached)
needs_go = "GO_IDs" not in df.columns or df["GO_IDs"].isna().all()
if needs_go:
    go = fetch_uniprot_go_annotations(df["Entry"].tolist(), cache_path=str(GO_CACHE))
    df = add_go_annotations(df, go)

if "Subcellular location [CC]_x" in df.columns:
    df["Subcellular location [CC]"] = df["Subcellular location [CC]_x"].fillna(
        df["Subcellular location [CC]_y"]
    )
    df = df.drop(columns=["Subcellular location [CC]_x", "Subcellular location [CC]_y"])

df[["Entry", "GO_IDs"]].head(2)

Loading GO annotation cache: /workspaces/mem_prot_llps/data/uniprot_go_cache.csv


,Entry,GO_IDs
0,Q9Y6V0,GO:0005509; GO:0005522; GO:0005544; GO:0005856...
1,Q9Y566,GO:0005829; GO:0005886; GO:0007616; GO:0008306...


In [4]:
# pLLPS classification
df["pLLPS_Class"] = pd.cut(
    df["p(LLPS)"],
    bins=[-float("inf"), 0.4, 0.7, float("inf")],
    labels=["Low", "Medium", "High"],
).astype(str)

df["pLLPS_Class"].value_counts()

pLLPS_Class
Low       10797
High       6568
Medium     3001
Name: count, dtype: int64

In [5]:
# Functional categories (GO-based)
df = add_functional_categories(df, go_col="GO_IDs")
df = add_go_slim_categories(df, go_col="GO_IDs", output_col="GO_Slim_Categories")

df["Functional_Categories"].value_counts().head(10)

/workspaces/mem_prot_llps/data/go/go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
Added functional categories to 20366 proteins  (16890 total assignments)
/workspaces/mem_prot_llps/data/go/goslim_generic.obo: fmt(1.2) rel(go/2026-03-25/subsets/goslim_generic.owl) 206 Terms


Functional_Categories
[]                        9124
[Transcription Factor]    1267
[Transferase]             1044
[GPCR, Receptor]           806
[Transporter]              635
[Hydrolase]                619
[Adhesion]                 545
[Protease, Hydrolase]      526
[Oxidoreductase]           526
[Structural]               521
Name: count, dtype: int64

In [6]:
# Location & function: derive from GO annotations (no string parsing)
# Split GO IDs by namespace (BP/MF/CC) and create GO Slim mappings via GOATOOLS
from llps.functional import parse_go_ids, _load_go_dag, _load_go_slim_dag, map_go_ids_to_slim

go_dag = _load_go_dag()
slim_dag = _load_go_slim_dag()

def split_go_namespaces(go_value):
    go_list = parse_go_ids(go_value)
    bp = []
    mf = []
    cc = []
    for gid in go_list:
        if gid not in go_dag:
            continue
        ns = getattr(go_dag[gid], 'namespace', None)
        if ns == 'biological_process':
            bp.append(gid)
        elif ns == 'molecular_function':
            mf.append(gid)
        elif ns == 'cellular_component':
            cc.append(gid)
    return bp, mf, cc

df[['GO_BP', 'GO_MF', 'GO_CC']] = df['GO_IDs'].apply(lambda v: pd.Series(split_go_namespaces(v)))

# Map GO IDs to GO Slim terms (use goslim_generic; limit mapping by namespace)
def slim_names_for(go_list):
    slim_ids = map_go_ids_to_slim(go_list, go_dag=go_dag, slim_dag=slim_dag)
    return [slim_dag[sid].name for sid in slim_ids if sid in slim_dag]

# Function (use BP/MF), Location (use CC)
df['Function_Slim'] = df['GO_BP'].apply(lambda gs: slim_names_for(gs)) + df['GO_MF'].apply(lambda gs: slim_names_for(gs))
df['Location_Slim'] = df['GO_CC'].apply(lambda gs: slim_names_for(gs))

# Top-level term for plotting (first slim hit or 'Other')
df['Function_Top'] = df['Function_Slim'].apply(lambda l: l[0] if l else 'Other')
df['Location_Top'] = df['Location_Slim'].apply(lambda l: l[0] if l else 'Other')

df[["Entry", "GO_BP", "GO_MF", "GO_CC", "Function_Top", "Location_Top"]].head(5)


,Entry,GO_BP,GO_MF,GO_CC,Function_Top,Location_Top
0,Q9Y6V0,"[GO:0007010, GO:0016079, GO:0017157, GO:003007...","[GO:0005509, GO:0005522, GO:0005544, GO:000827...","[GO:0005856, GO:0014069, GO:0030424, GO:004520...",cytoskeleton organization,cytoskeleton
1,Q9Y566,"[GO:0007616, GO:0008306, GO:0030534, GO:003223...","[GO:0017124, GO:0030160, GO:0031877, GO:003525...","[GO:0005829, GO:0005886, GO:0014069, GO:001602...",cell junction organization,cytosol
2,Q9Y520,"[GO:0002244, GO:0034063]",[GO:0003723],"[GO:0005829, GO:0010494, GO:0016020]",cell differentiation,cytosol
3,Q9Y4H2,"[GO:0002053, GO:0002903, GO:0006006, GO:000716...","[GO:0005068, GO:0005158, GO:0019903, GO:001990...","[GO:0005737, GO:0005829, GO:0005886]",carbohydrate metabolic process,cytosol
4,Q9Y3S1,"[GO:0006468, GO:0006974, GO:0008285, GO:003555...","[GO:0004674, GO:0005524, GO:0106310]","[GO:0005737, GO:0005829, GO:0005886]",transferase activity,cytosol


In [7]:
# TMD count (reads from local cache â€” run scripts/analysis/enrich_dataset_with_tmd.py to populate)
if CACHE.exists():
    tm = fetch_uniprot_tm_annotations(df["Entry"].tolist(), cache_path=str(CACHE))
    df = add_tmd_count(df, tm)
else:
    df["TMD_count"] = 0
    print("âš  TM cache not found â€” TMD_count set to 0")

df["TMD_count"].value_counts().sort_index().head(10)

Loading TM annotation cache: /workspaces/mem_prot_llps/data/uniprot_tm_cache.csv
TMD_count added: 5218 proteins have ≥1 membrane domain (max=38)


TMD_count
0    15148
1     2376
2      322
3      153
4      409
5       88
6      206
7      983
8      111
9       64
Name: count, dtype: int64

In [8]:
# Remove legacy string-based membrane/location inference — using GO-based Location_Top instead
# If TMD_count is still useful, preserve it but do not infer compartments from text parsing
df['Is_Membrane'] = df.get('TMD_count', 0).fillna(0).astype(int) > 0

# Use GO-derived Location_Top as the compartment label for plotting
df['Compartment'] = df['Location_Top']

print(df['Is_Membrane'].sum(), 'membrane proteins (TMD_count>0)')
df['Compartment'].value_counts()


5218 membrane proteins (TMD_count>0)


Compartment
nucleus                          5436
Other                            3532
plasma membrane                  2552
extracellular region             1985
cytosol                          1480
nucleoplasm                      1288
mitochondrion                    1029
endoplasmic reticulum             600
endosome                          382
Golgi apparatus                   374
microtubule organizing center     274
organelle                         257
cytoskeleton                      206
lysosome                          184
nucleolus                         165
nuclear envelope                  119
cilium                            118
extracellular matrix               91
cytoplasmic vesicle                79
nuclear chromosome                 66
peroxisome                         58
chromosome                         34
lipid droplet                      33
vacuole                            20
ribosome                            4
Name: count, dtype: int64

In [9]:
# Drop legacy location columns (GO-based columns replace these)
_drop = [
    "All_Functional_Groups", "Function Categories", "Functional Group", "Functional Slim",
    "Transmembrane", "Intramembrane",
    "Subcellular location [CC]_x", "Subcellular location [CC]_y",
    "Location Categories", "Location_IDs", "Location Primary",
    "Location_SL_IDs", "Location_Names",
    *[c for c in df.columns if c.startswith("Is_") and c != "Is_Membrane"],
]
df = df.drop(columns=[c for c in _drop if c in df.columns])

# Save
OUT.parent.mkdir(exist_ok=True)
df.to_csv(OUT, index=False)
APP_OUT.parent.mkdir(exist_ok=True)
df.to_csv(APP_OUT, index=False)

print(f"Saved {len(df):,} proteins  |  {df.shape[1]} columns")
print(df.columns.tolist())


Saved 20,366 proteins  |  25 columns
['Entry', 'Entry name', 'Protein names', 'p(LLPS)', 'n(DPR=> 25)', 'Organism', 'Length', 'Function [CC]', 'Involvement in disease', 'Cross-reference (PDB)', 'GO_IDs', 'Subcellular location [CC]', 'pLLPS_Class', 'Functional_Categories', 'GO_Slim_Categories', 'GO_BP', 'GO_MF', 'GO_CC', 'Function_Slim', 'Location_Slim', 'Function_Top', 'Location_Top', 'TMD_count', 'Is_Membrane', 'Compartment']
